<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9; border-right: 5px solid #20639b; padding: 8px 18px;">
  <h1 style="color:#173f5f; margin-bottom:4px;">نوت‌بوک 10: پروژهٔ کوچک پایتون: مدیریت موجودی و سفارش</h1>
  <p><b>سطح:</b> متوسط و کاربردی &nbsp; | &nbsp; <b>روش مطالعه:</b> توضیح کوتاه ← اجرای مثال ← تغییر مثال ← حل تمرین</p>
  <h3>هدف‌های یادگیری</h3>
  <ul><li>ترکیب ساختار داده، شرط، حلقه، تابع و کلاس</li>
<li>اعتبارسنجی قواعد سفارش</li>
<li>تولید گزارش خوانا</li>
<li>تفکیک مسئولیت‌ها در یک برنامهٔ کوچک</li></ul>
  <p style="background:#eef6fb; padding:10px; border-radius:8px;">همهٔ سلول‌ها را به‌ترتیب اجرا کنید. برای یادگیری بهتر، مقدار ورودی‌ها را تغییر دهید و نتیجه را پیش‌بینی کنید.</p>
</div>

<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">صورت پروژه</h2>
  <p>یک فروشگاه کوچک داریم. محصولات موجودی و قیمت دارند. سفارش باید بررسی شود، موجودی را کم کند و فاکتور بسازد. هدف این پروژه مشاهدهٔ ارتباط مفاهیم قبلی در یک مثال واحد است.</p>
</div>

<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">۱. مدل محصول</h2>
  <p>کلاس محصول از ایجاد قیمت یا موجودی منفی جلوگیری می‌کند و ارزش موجودی را محاسبه می‌کند.</p>
</div>

In [1]:
# Define the product entity
from dataclasses import dataclass

@dataclass
class Product:
    sku: str
    name: str
    price: float
    stock: int

    def __post_init__(self):
        if self.price < 0 or self.stock < 0:
            raise ValueError("price and stock must be non-negative")

    def inventory_value(self):
        return self.price * self.stock


<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">۲. انبار</h2>
  <p>انبار محصولات را با SKU نگه می‌دارد. متد فروش، وجود کالا و کافی‌بودن موجودی را بررسی می‌کند.</p>
</div>

In [2]:
# Manage products and stock changes
class Inventory:
    def __init__(self):
        self.products = {}

    def add_product(self, product):
        if product.sku in self.products:
            raise ValueError("duplicate SKU")
        self.products[product.sku] = product

    def sell(self, sku, quantity):
        if quantity <= 0:
            raise ValueError("quantity must be positive")
        if sku not in self.products:
            raise KeyError("unknown SKU")

        product = self.products[sku]
        if quantity > product.stock:
            raise ValueError("insufficient stock")

        product.stock -= quantity
        return product.price * quantity

    def low_stock(self, threshold=3):
        return [
            product for product in self.products.values()
            if product.stock <= threshold
        ]


<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">۳. ساخت دادهٔ نمونه</h2>
  <p>سه محصول به انبار اضافه می‌کنیم. در برنامهٔ واقعی این داده می‌تواند از فایل یا پایگاه داده دریافت شود.</p>
</div>

In [3]:
# Seed a small inventory
inventory = Inventory()
inventory.add_product(Product("P101", "Keyboard", 1_850_000, 6))
inventory.add_product(Product("P102", "Mouse", 920_000, 10))
inventory.add_product(Product("P103", "Monitor", 8_500_000, 2))

print("Product count:", len(inventory.products))


Product count: 3


<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">۴. پردازش سفارش</h2>
  <p>تابع زیر هر ردیف سفارش را پردازش می‌کند. ردیف‌های نامعتبر در گزارش خطا ثبت می‌شوند و برنامه ادامه پیدا می‌کند.</p>
</div>

In [4]:
# Process order lines and collect validation errors
def process_order(inventory, order_lines, discount_percent=0):
    if not 0 <= discount_percent <= 100:
        raise ValueError("discount must be between 0 and 100")

    successful_lines = []
    errors = []

    for line in order_lines:
        sku = line.get("sku")
        quantity = line.get("quantity", 0)
        try:
            line_total = inventory.sell(sku, quantity)
        except (KeyError, ValueError) as error:
            errors.append({"sku": sku, "error": str(error)})
            continue

        successful_lines.append({
            "sku": sku,
            "quantity": quantity,
            "line_total": line_total,
        })

    subtotal = sum(line["line_total"] for line in successful_lines)
    discount = subtotal * discount_percent / 100
    return {
        "items": successful_lines,
        "subtotal": subtotal,
        "discount": discount,
        "final_total": subtotal - discount,
        "errors": errors,
    }


<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">۵. اجرای سناریو</h2>
  <p>یک سفارش شامل دو ردیف معتبر و یک ردیف با موجودی ناکافی را اجرا می‌کنیم.</p>
</div>

In [5]:
# Run a realistic order scenario
order_lines = [
    {"sku": "P101", "quantity": 2},
    {"sku": "P102", "quantity": 1},
    {"sku": "P103", "quantity": 5},
]

invoice = process_order(inventory, order_lines, discount_percent=10)

print("Successful lines:")
for item in invoice["items"]:
    print(item)

print("Errors:", invoice["errors"])
print(f"Final total: {invoice['final_total']:,.0f}")


Successful lines:
{'sku': 'P101', 'quantity': 2, 'line_total': 3700000}
{'sku': 'P102', 'quantity': 1, 'line_total': 920000}
Errors: [{'sku': 'P103', 'error': 'insufficient stock'}]
Final total: 4,158,000


<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">۶. گزارش موجودی</h2>
  <p>پس از سفارش، ارزش کل موجودی و کالاهای کم‌موجودی گزارش می‌شوند.</p>
</div>

In [6]:
# Build an inventory summary
inventory_value = sum(
    product.inventory_value()
    for product in inventory.products.values()
)
low_stock_names = [product.name for product in inventory.low_stock()]

print(f"Inventory value: {inventory_value:,.0f}")
print("Low-stock products:", low_stock_names)


Inventory value: 32,680,000
Low-stock products: ['Monitor']


<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">تمرین کوتاه</h2>
  <div style="background:#fff7df; border:1px solid #f0c36d; padding:12px; border-radius:8px;">قابلیت محاسبهٔ ۹٪ مالیات را به خروجی تابع سفارش اضافه کنید و مبلغ قابل پرداخت را پس از تخفیف و مالیات به دست آورید.</div>
</div>

<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">پاسخ پیشنهادی</h2>
  <p>ابتدا پاسخ خودتان را بنویسید؛ سپس این سلول را اجرا و مقایسه کنید.</p>
</div>

In [7]:
# Extend the invoice without changing inventory state again
tax_rate = 0.09
taxable_amount = invoice["final_total"]
tax = taxable_amount * tax_rate
payable = taxable_amount + tax

invoice_with_tax = {
    **invoice,
    "tax": tax,
    "payable": payable,
}

print(f"Tax: {invoice_with_tax['tax']:,.0f}")
print(f"Payable: {invoice_with_tax['payable']:,.0f}")


Tax: 374,220
Payable: 4,532,220


<div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.9;">
  <h2 style="color:#173f5f;">مسیر توسعه</h2>
  <p>گام بعدی می‌تواند ذخیرهٔ فاکتور در JSON، خواندن محصولات از CSV یا نوشتن آزمون برای قوانین موجودی باشد.</p>
</div>